# Figure 2. Perturbation space exploration

Notebook including main analysis performed in this study to validate that the embedding was capturing the perturbaiton effect. 
- 2D representation of embedding spaces (pre-trained, full fine-tuned, HQ fine-tuned)
- Pairwise cosine distance distributions for scGPT embedings grouped bu same cell, same perturbation and random
- Intra-coherence analysis comparing the distance between samples of the same pertubation in the embedding space vs GEx. 
- Cluster coherence metrics (ARI)
- Explore local neigborhoods with Hit@100

# Import packages and load data

In [ ]:
# IMPORT NEEDED PACKAGES 
import pandas as pd
import numpy as np

from tqdm import tqdm
import pickle
from collections import Counter

import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc
import h5py

path_save = './LINCS_scGPT_embeddings/05_Figures/'



# 1. UMAP 

In [ ]:
# PATHS of interest

name_adata_FT = './LINCS_scGPT_embeddings/02_Obtain_Embeddings/embeddings_full.h5ad'# 20 epochs FT classification all data
name_adata_original = './LINCS_scGPT_embeddings/02_Obtain_Embeddings/embeddings_pretrained.h5ad' # Original scGPT
name_adata_selected = './LINCS_scGPT_embeddings/02_Obtain_Embeddings/embeddings_HQ.h5ad' # 50 epochs HQ data FT classification

In [ ]:
# Load adata
adata_FT = sc.read(name_adata_FT)
adata_original = sc.read(name_adata_original)
adata_selected = sc.read(name_adata_selected)

In [ ]:
cells = adata_original.obs.cell_iname.value_counts().index.tolist()
colors =sns.color_palette("pastel") +sns.color_palette('Set2') + sc.pl.palettes.default_102 + sns.color_palette("pastel", 89) + sns.color_palette('flare')+sns.color_palette('Set3') + sns.color_palette('Paired') + sns.color_palette('husl') + sns.color_palette('tab20')
cell_id_color = dict(zip(cells, colors)) 


In [ ]:
adata_FT.obs['color'] = [cell_id_color[i] for i in adata_FT.obs.cell_iname]
adata_original.obs['color'] = [cell_id_color[i] for i in adata_original.obs.cell_iname]
adata_selected.obs['color'] = [cell_id_color[i] for i in adata_selected.obs.cell_iname]



In [ ]:
fig, axs = plt.subplots(1,3, figsize=(12,5), dpi=300, facecolor='white')
axs = axs.flatten()
axs[0].scatter(adata_original.obsm['X_umap'][:, 0], adata_original.obsm['X_umap'][:, 1],color = adata_original.obs.color, s = 0.01, alpha = 0.2)
axs[0].set_title('Pre-trained scGPT\n%s signatures'%adata_original.shape[0])
axs[0].axis('off')

axs[1].scatter(adata_FT.obsm['X_umap'][:, 0], adata_FT.obsm['X_umap'][:, 1],color = adata_FT.obs.color, s = 0.01, alpha = 0.2)
axs[1].set_title('Fine-tuned scGPT\n%s signatures'%adata_FT.shape[0])
axs[1].axis('off')

axs[2].scatter(adata_selected.obsm['X_umap'][:, 0], adata_selected.obsm['X_umap'][:, 1],color = adata_selected.obs.color, s = 0.01, alpha = 0.2)
axs[2].set_title('Fine-tuned scGPT (HQ data)\n%s signatures'%adata_selected.shape[0])
axs[2].axis('off')


fig.suptitle('UMAP of scGPT Embeddings')
plt.tight_layout()


# 2. Distribution of distances

## Functions to select pairs of embeddings to compare and compute distances

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from tqdm import tqdm
from scipy.spatial.distance import cosine


def select_feature(adata, feature, ref="pert_id", cmp=True):
    """
    Select samples and create a metadata dataframe with the feature of interest.

    If cmp=True, only compound perturbations are kept: pert_type == 'trt_cp'.
    """
    if cmp:
        adata_feature = adata[adata.obs["pert_type"] == "trt_cp"].copy()
    else:
        adata_feature = adata.copy()

    feature_order = adata_feature.obs[feature].values.astype(str)
    ref_order = adata_feature.obs[ref].values.astype(str)

    df_meta = pd.DataFrame({
        ref: ref_order,
        feature: feature_order,
    })

    return adata_feature, df_meta


def select_pairs_unique_feature(
    df_meta,
    feature,
    exclusion_feature,
    sample_size=1000,
    seed=0,
):
    """
    Select random pairs of samples that share `feature`
    but have different values for `exclusion_feature`.
    """
    rng = np.random.default_rng(seed)
    pairs = []

    for f_value, f_group in tqdm(df_meta.groupby(feature), desc=f"Selecting pairs by {feature}"):
        ex_group = {
            ex_value: group.index.values
            for ex_value, group in f_group.groupby(exclusion_feature)
        }

        ex_values = list(ex_group.keys())

        if len(ex_values) > 1:
            for _ in range(sample_size):
                ex_f1, ex_f2 = rng.choice(ex_values, 2, replace=False)

                if len(ex_group[ex_f1]) > 0 and len(ex_group[ex_f2]) > 0:
                    idx1 = rng.choice(ex_group[ex_f1])
                    idx2 = rng.choice(ex_group[ex_f2])
                    pairs.append(sorted([idx1, idx2]))

    if len(pairs) == 0:
        return np.empty((0, 2), dtype=int)

    pairs_unique = (
        pd.DataFrame(np.array(pairs), columns=["sample1", "sample2"])
        .drop_duplicates()
        .to_numpy()
    )

    return pairs_unique


def select_random_pairs_diff_features(
    df_meta,
    feature1,
    feature2,
    sample_size=1000,
    seed=0,
):
    """
    Select random pairs where both feature1 and feature2 are different.
    """
    rng = np.random.default_rng(seed)
    indices = df_meta.index.values
    pairs = []

    for _ in tqdm(range(sample_size), desc="Selecting random pairs with different features"):
        idx1, idx2 = rng.choice(indices, 2, replace=False)

        if (
            df_meta.loc[idx1, feature1] != df_meta.loc[idx2, feature1]
            and df_meta.loc[idx1, feature2] != df_meta.loc[idx2, feature2]
        ):
            pairs.append(sorted([idx1, idx2]))

    if len(pairs) == 0:
        return np.empty((0, 2), dtype=int)

    pairs_unique = (
        pd.DataFrame(np.array(pairs), columns=["sample1", "sample2"])
        .drop_duplicates()
        .to_numpy()
    )

    return pairs_unique


def compute_distances(adata, pairs, obsm_key="X_scGPT"):
    """
    Compute cosine distances between selected pairs.
    """
    distances = [
        cosine(adata.obsm[obsm_key][i], adata.obsm[obsm_key][j])
        for i, j in tqdm(pairs, desc="Computing distances")
    ]

    return np.array(distances)


def get_all_pair_distances_for_adata(
    adata,
    cell_feature="cell_id",
    pert_feature="pert_id",
    obsm_key="X_scGPT",
    sample_size=1000,
    seed=0,
    cmp=True,
):
    adata_feature, df_meta = select_feature(
        adata,
        feature=cell_feature,
        ref=pert_feature,
        cmp=cmp,
    )

    pairs_cell = select_pairs_unique_feature(
        df_meta=df_meta,
        feature=cell_feature,
        exclusion_feature=pert_feature,
        sample_size=sample_size,
        seed=seed,
    )

    pairs_drug = select_pairs_unique_feature(
        df_meta=df_meta,
        feature=pert_feature,
        exclusion_feature=cell_feature,
        sample_size=sample_size,
        seed=seed,
    )

    pairs_random = select_random_pairs_diff_features(
        df_meta=df_meta,
        feature1=cell_feature,
        feature2=pert_feature,
        sample_size=sample_size,
        seed=seed,
    )

    distances_cell = compute_distances(adata_feature, pairs_cell, obsm_key)
    distances_drug = compute_distances(adata_feature, pairs_drug, obsm_key)
    distances_random = compute_distances(adata_feature, pairs_random, obsm_key)

    distances = {
        "Same cell": distances_cell,
        "Same pert": distances_drug,
        "Diff cell / pert": distances_random,
    }

    pairs = {
        "Same cell": pairs_cell,
        "Same pert": pairs_drug,
        "Diff cell / pert": pairs_random,
    }

    return adata_feature, df_meta, pairs, distances

## Compute distances 

In [ ]:
# Load adatas

feature = "cell_iname"          # same cell line
exclusion_feature = "pert_id"  # different compounds

adata_original_sub, meta_original, pairs_original, distances_original = get_all_pair_distances_for_adata(
    adata_original,
    cell_feature="cell_id",
    pert_feature="pert_id",
    obsm_key="X_scGPT",
    sample_size=1000,
    seed=0,
    cmp=False
)

adata_FT_sub, meta_FT, pairs_FT, distances_FT = get_all_pair_distances_for_adata(
    adata_FT,
    cell_feature="cell_id",
    pert_feature="pert_id",
    obsm_key="X_scGPT",
    sample_size=1000,
    seed=0,
    cmp=False
)

adata_selected_sub, meta_selected, pairs_selected, distances_selected = get_all_pair_distances_for_adata(
    adata_selected,
    cell_feature="cell_iname",
    pert_feature="pert_id",
    obsm_key="X_scGPT",
    sample_size=1000,
    seed=0,
    cmp = False
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


def plot_distance_comparison(
    distances_original,
    distances_FT,
    distances_selected,
    title="Pairwise cosine distance between scGPT embeddings",
    save_path=None,
):
    fig, axs = plt.subplots(1, 3, figsize=(12, 4), dpi=200, facecolor="white")
    axs = axs.flatten()

    panels = {
        "Original": distances_original,
        "FT": distances_FT,
        "Selected": distances_selected,
    }

    colors = {
        "Same cell": sns.color_palette("pastel")[4],
        "Same pert": sns.color_palette("pastel")[0],
        "Diff cell / pert": sns.color_palette("pastel")[1],
    }

    for ax, (panel_name, distances_dict) in zip(axs, panels.items()):
        sns.kdeplot(
            distances_dict["Same cell"],
            label="Same cell",
            color=colors["Same cell"],
            ax=ax,
            fill=True,
            alpha=0.6,
            linewidth=2,
        )

        sns.kdeplot(
            distances_dict["Same pert"],
            label="Same pert",
            color=colors["Same pert"],
            ax=ax,
            fill=True,
            alpha=0.6,
            linewidth=2,
        )

        sns.kdeplot(
            distances_dict["Diff cell / pert"],
            label="Diff cell / pert",
            color=colors["Diff cell / pert"],
            ax=ax,
            fill=True,
            alpha=0.6,
            linewidth=2,
        )

        ax.set_title(panel_name)
        ax.set_xlabel("Cosine distance")
        ax.spines["right"].set_visible(False)
        ax.spines["top"].set_visible(False)

    axs[1].set_ylabel("")
    axs[2].set_ylabel("")

    handles, labels = axs[0].get_legend_handles_labels()
    legend = fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
        frameon=False,
        fontsize=10,
        bbox_to_anchor=(0.5, 1.08),
    )

    for handle in legend.legend_handles:
        handle.set_linewidth(1)

    fig.suptitle(title, y=1.18)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, format="svg", bbox_inches="tight")

    plt.show()

In [ ]:
plot_distance_comparison(
    distances_original,
    distances_FT,
    distances_selected,
)

# 3. Intraclass agreement

In [ ]:
from scipy.spatial.distance import pdist

def compute_distances(matrix, labels, metric='cosine', scale='percentile'):
    group_distances = {}
    all_distances = pdist(matrix, metric=metric)

    # Space-specific stats
    min_dist, max_dist = all_distances.min(), all_distances.max()
    mean_dist, std_dist = np.mean(all_distances), np.std(all_distances)

    # For percentile mode
    if scale == 'percentile':
        all_sorted = np.sort(all_distances)  
        n_all = len(all_sorted) 


    for annotation in np.unique(labels):
        group = matrix[labels == annotation]
        if len(group) > 1:
            distances = pdist(group, metric=metric)
            if scale == 'minmax':
                distances = (distances - min_dist) / (max_dist - min_dist + 1e-8)
            elif scale == 'zscore':
                distances = (distances - mean_dist) / (std_dist + 1e-8)
            elif scale == 'average':
                distances = (distances - mean_dist) / (mean_dist + 1e-8)
            elif scale == 'percentile':
                distances = np.searchsorted(all_sorted, distances, side="right") / n_all
                
            avg_distance = distances.mean()
            max_distance = distances.max()
        else:
            avg_distance = max_distance = 0
            distances = []
        group_distances[annotation] = (avg_distance, max_distance, distances, len(group))
    return group_distances


In [ ]:
def select_data(adata, label, n_groups, samples_per_group, random_state=42):

    labels = adata.obs[label]
    counts = labels.value_counts()
    valid_groups = counts[counts >= 2].index

    adata_valid = adata[adata.obs[label].isin(valid_groups)].copy()
    obs_df = adata_valid.obs[[label]].copy()
    obs_df["index"] = obs_df.index

    rng = np.random.default_rng(random_state)
    selected_groups = rng.choice(valid_groups, size=n_groups, replace=False)

    sampled_idx = (
        obs_df[obs_df[label].isin(selected_groups)]
        .groupby(label)
        .apply(lambda x: x.sample(n=min(samples_per_group, len(x)), random_state=random_state))
        .reset_index(drop=True)["index"]
    )

    return adata_valid[sampled_idx].X, adata_valid[sampled_idx].obsm["X_scGPT"], adata_valid[sampled_idx].obs[label].values



In [ ]:
from scipy.stats import ttest_ind, mannwhitneyu
def cluster_evaluation(m1, m2, labels, names=['M1', 'M2'], metric='cosine', scale='percentile', test='mannwhitney', title='Cluster Evaluation', use_robust_y = True, y_quantile = 0.95):
    # Compute distances for both descriptors
    m1_distances = compute_distances(m1, labels, metric=metric, scale=scale)
    m2_distances = compute_distances(m2, labels, metric=metric, scale=scale)

    # Prepare data for plotting
    annotations = list(set(m1_distances.keys()) & set(m2_distances.keys()))  # Common annotations
    x_vals = []
    y_vals = []
    sizes = []
    colors = []
    pvals = []

    for annotation in annotations:
        avg_m1, max_m1, distances_m1, count_m1 = m1_distances[annotation]
        avg_m2, max_m2, distances_m2, count_m2 = m2_distances[annotation]
        
        x_diff = avg_m1 - avg_m2
        x_vals.append(x_diff)  # Difference in average distances
        
        if use_robust_y and len(distances_m1) > 0 and len(distances_m2) > 0:
            y1 = np.quantile(distances_m1, y_quantile)
            y2 = np.quantile(distances_m2, y_quantile)
            y_vals.append(max(y1, y2))
        else:
            y_vals.append(max(max_m1, max_m2))  # Maximal distance
        sizes.append(count_m1 + count_m2)  # Total number of molecules

        # Statistical test
        if len(distances_m1) > 1 and len(distances_m2) > 1:
            try:
                if test=='ttest':
                    stat, pval = ttest_ind(distances_m1, distances_m2, equal_var=False)
                elif test=='mannwhitney':
                    # Alternative: 
                    stat, pval = mannwhitneyu(distances_m1, distances_m2, alternative='two-sided')
            except Exception:
                pval = 1.0
        else:
            pval = 1.0

        pvals.append(pval)
        colors.append(-np.log10(pval + 1e-10))  # avoid log(0)
        #colors.append(x_diff)  # Use x_diff for coloring

    # Compute significance mask
    significant = np.array(pvals) < 0.05

    # Optional: thicker border if significant
    edge_widths = [1.5 if sig else 0.5 for sig in significant]

    return x_vals, y_vals, sizes, colors, edge_widths

To do this analysis, the data should be downloaded from https://clue.io/data/CMap2020#LINCS2020 and add the level 3 data to the `adata.X`

In [ ]:
GEX_original_pert, EMB_original_pert, labels_original_pert = select_data(adata_original, 'pert_id', n_groups=5000, samples_per_group=10, random_state=2)
GEX_FT_pert, EMB_FT_pert, labels_FT_pert = select_data(adata_FT, 'pert_id', n_groups=5000, samples_per_group=10, random_state=2)
GEX_selected_pert, EMB_selected_pert, labels_selected_pert = select_data(adata_selected, 'pert_id', n_groups=5000, samples_per_group=10, random_state=2)

In [ ]:
x_vals_original, y_vals_original, sizes_original, colors_original, edge_widths_original = cluster_evaluation(GEX_original_pert, EMB_original_pert, labels_original_pert, ['GEx', 'Pre-trained'], scale='percentile', metric='cosine', test='mannwhitney', use_robust_y=True, y_quantile=0.95)
x_vals_FT, y_vals_FT, sizes_FT, colors_FT, edge_widths_FT = cluster_evaluation(GEX_FT_pert, EMB_FT_pert, labels_FT_pert, ['GEx', 'Fine-tuned'], scale='percentile', metric='cosine', test='mannwhitney', use_robust_y=True, y_quantile=0.95)
x_vals_selected, y_vals_selected, sizes_selected, colors_selected, edge_widths_selected = cluster_evaluation(GEX_selected_pert, EMB_selected_pert, labels_selected_pert, ['GEx', 'Fine-tuned HQ'], scale='percentile', metric='cosine', test='mannwhitney', use_robust_y=True, y_quantile=0.95)

intra_data = {
    "original": {
        "x": x_vals_original,
        "y": y_vals_original,
        "sizes": sizes_original,
        "colors": colors_original,
        "edge_widths": edge_widths_original,
    },
    "FT": {
        "x": x_vals_FT,
        "y": y_vals_FT,
        "sizes": sizes_FT,
        "colors": colors_FT,
        "edge_widths": edge_widths_FT,
    },
    "selected": {
        "x": x_vals_selected,
        "y": y_vals_selected,
        "sizes": sizes_selected,
        "colors": colors_selected,
        "edge_widths": edge_widths_selected,
    },
}

# 4. Clustering and Adjusted Rand Index

To compute clusters for PCA or GEx, the data should be downloaded from https://clue.io/data/CMap2020#LINCS2020 and add the level 3 data to the `adata.X`


Also you can download the values of all clustering metrics from the Supplementary Table 1 of the paper. 

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

def pca_adata (adata, n_components): 
    pca = PCA(n_components=n_components)
    adata.obsm['X_pca_raw'] = pca.fit_transform(adata.X)
    return adata

kmeans = MiniBatchKMeans(n_clusters=int(np.sqrt(adata_FT.shape[0])/2), random_state=42, batch_size=10000)
labels_emb_FT = kmeans.fit_predict(adata_FT.obsm['X_scGPT'])
adata_FT.obs['labels_emb'] = labels_emb_FT

# labels_GEx_FT = kmeans.fit_predict(adata_FT.X) 
# labels_PCA_FT = kmeans.fit_predict(adata_FT.obsm['X_pca_raw'])

# adata_FT.obs['labels_GEx'] = labels_GEx_FT
# adata_FT.obs['labels_PCA'] = labels_PCA_FT

kmeans_ori = MiniBatchKMeans(n_clusters=int(np.sqrt(adata_original.shape[0])/2), random_state=42, batch_size=10000)
labels_emb_ori = kmeans_ori.fit_predict(adata_original.obsm['X_scGPT'])
adata_original.obs['labels_emb'] = labels_emb_ori

# labels_GEx_ori = kmeans_ori.fit_predict(adata_original.X) 
# labels_PCA_ori = kmeans_ori.fit_predict(adata_original.obsm['X_pca_raw'])

# adata_original.obs['labels_GEx'] = labels_GEx_ori
# adata_original.obs['labels_PCA'] = labels_PCA_ori

kmeans_sel = MiniBatchKMeans(n_clusters=int(np.sqrt(adata_selected.shape[0])/2), random_state=42, batch_size=10000)
labels_emb_sel = kmeans_sel.fit_predict(adata_selected.obsm['X_scGPT'])
adata_selected.obs['labels_emb'] = labels_emb_sel

# labels_GEx_sel = kmeans_sel.fit_predict(adata_selected.X) 
# labels_PCA_sel = kmeans_sel.fit_predict(adata_selected.obsm['X_pca_raw'])

# adata_selected.obs['labels_GEx'] = labels_GEx_sel
# adata_selected.obs['labels_PCA'] = labels_PCA_sel

In [ ]:
# Create a replicate group ID based on compound, dose, cell, etc.
adata_original.obs["replicate_group"] = (
    adata_original.obs["pert_id"].astype(str) + "_" + adata_original.obs["pert_dose"].astype(float).round(2).astype(str) + "_" + adata_original.obs["cell_iname"].astype(str) + "_" + adata_original.obs["pert_time"].astype(float).round(2).astype(str)
)

# Create a replicate group ID based on compound, dose, cell, etc.
adata_FT.obs["replicate_group"] = (
    adata_FT.obs["pert_id"].astype(str) + "_" + adata_FT.obs["pert_dose"].astype(float).round(2).astype(str) + "_" + adata_FT.obs["cell_iname"].astype(str) + "_" + adata_FT.obs["pert_time"].astype(float).round(2).astype(str)
)

# Create a replicate group ID based on compound, dose, cell, etc.
adata_selected.obs["replicate_group"] = (
    adata_selected.obs["pert_id"].astype(str) + "_" + adata_selected.obs["pert_dose"].astype(float).round(2).astype(str) + "_" + adata_selected.obs["cell_iname"].astype(str) + "_" + adata_selected.obs["pert_time"].astype(float).round(2).astype(str)
)

adata_selected.obs["pert_dose"] = adata_selected.obs["pert_dose"].astype(float).fillna(-666)

adata_original.obs['pert_dose'] = adata_original.obs['pert_dose'].astype(float).round(2).astype(str)
adata_FT.obs['pert_dose'] = adata_FT.obs['pert_dose'].astype(float).round(2).astype(str)
adata_selected.obs['pert_dose'] = adata_selected.obs['pert_dose'].astype(float).round(2).astype(str)


adata_original.obs['pert_time'] = adata_original.obs['pert_time'].astype(float).round(2).astype(str)
adata_FT.obs['pert_time'] = adata_FT.obs['pert_time'].astype(float).round(2).astype(str)
adata_selected.obs['pert_time'] = adata_selected.obs['pert_time'].astype(float).round(2).astype(str)



In [ ]:
representations = {
                  'Original emb': adata_original.obs['labels_emb'],
                  'FT emb': adata_FT.obs['labels_emb'],
                  'Selected emb': adata_selected.obs['labels_emb'],
                 }

conditions = ['pert_id', 'replicate_group','pert_dose', 'pert_time', 'cell_iname', 'pert_type']

In [ ]:
from sklearn.metrics import adjusted_rand_score
results = []

for name, clusters in tqdm(representations.items()):
    if name in ['Original Raw GEx', 'Original emb', 'Original PCA']:
        adata = adata_original.copy()
    elif name in ['FT Raw GEx', 'FT emb', 'FT PCA']:
        adata = adata_FT.copy()
    elif name in ['Selected Raw GEx', 'Selected emb', 'Selected PCA']:
        adata = adata_selected.copy()
    else:
        continue  # skip unknown representations

    for label in conditions: 
        ari = adjusted_rand_score(adata.obs[label], clusters)
        results.append({'representation': name, 'label': label, 'ARI': ari})

df_ari = pd.DataFrame(results)
df_ari['type'] = df_ari.representation.str.split(' ').str[-1]
df_ari['model'] = df_ari.representation.str.split(' ').str[0]
df_ari["model"] = pd.Categorical(df_ari["model"], categories=["Original", "FT", "Selected"], ordered=True)
df_ari["source"] = pd.Categorical(df_ari["type"], categories=["GEx","PCA", "emb"], ordered=True)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


################# figure 2D ####################
label = "cell_iname"  # or "pert_id"
fig, axes = plt.subplots(figsize=(4, 4))

subset = ari[ari["label"] == label].copy()

# ✅ Rename in the HUE column (source), not model
subset["source"] = subset["source"].astype(str).replace({"emb": "scGPT"})
subset["model"] = subset["model"].astype(str).replace({"Original": "Pre-trained", "FT": "Fine-tuned", 'Selected': 'Fine-tuned HQ'})


# Bars order within each x-group is controlled by hue_order:
hue_order = ["GEx", "PCA", "scGPT"]

# Palette must map the HUE levels:
palette = {
    "GEx": "#FF6961",                 # pastel red
    "PCA": "#77DD77",                 # pastel green
    "scGPT": sns.color_palette("pastel")[9]  # pastel violet
}

sns.barplot(
    data=subset,
    x="model", y="ARI", hue="source",
    hue_order=hue_order,
    palette=palette,
    edgecolor="black",
    ax=axes,
    zorder=3
)

axes.set_title("Clustering performance by cell id")
axes.set_ylim(0, subset["ARI"].max() * 1.2)
axes.set_ylabel("Adjusted Rand Index")
axes.set_xlabel("Model")
axes.grid(axis="y", linestyle="--", zorder=2)
axes.legend(title="Data type")

plt.tight_layout()

################# figure 2E ####################
label = "pert_id"  # or "pert_id"
fig, axes = plt.subplots(figsize=(4, 4))

subset = ari[ari["label"] == label].copy()

# ✅ Rename in the HUE column (source), not model
subset["source"] = subset["source"].astype(str).replace({"emb": "scGPT"})
subset["model"] = subset["model"].astype(str).replace({"Original": "Pre-trained", "FT": "Fine-tuned", 'Selected': 'Fine-tuned HQ'})

# Bars order within each x-group is controlled by hue_order:
hue_order = ["GEx", "PCA", "scGPT"]

# Palette must map the HUE levels:
palette = {
    "GEx": "#FF6961",                 # pastel red
    "PCA": "#77DD77",                 # pastel green
    "scGPT": sns.color_palette("pastel")[9]  # pastel violet
}

sns.barplot(
    data=subset,
    x="model", y="ARI", hue="source",
    hue_order=hue_order,
    palette=palette,
    edgecolor="black",
    ax=axes,
    zorder=3
)

axes.set_title("Clustering performance by pert id")
axes.set_ylim(0, subset["ARI"].max() * 1.2)
axes.set_ylabel("Adjusted Rand Index")
axes.set_xlabel("Model")
axes.grid(axis="y", linestyle="--", zorder=2)
axes.legend(title="Data type")

plt.tight_layout()



# 5. Compound recovery (Hit@100)

In [ ]:
## Distances and indices
# =============================================================================
# You need to run ./LINCS_scGPT_embeddings/03_Distance_Integration/NN_calculation.py using: 
# - reference: embeddings of compounds (trt_cp)
# - query: embeddings of compounds (trt_cp)

In [ ]:
sig2pert = dict(zip(adata_FT.obs_names, adata_FT.obs['pert_id']))

def overall_hit(sig_id_order, indices, k=[1, 5, 10, 100]):

    sig_pert = np.array([sig2pert[i] for i in sig_id_order])
    precisions = {}
    for k in tqdm([1, 5, 10, 100]):
        # Take top-k neighbors (exclude the first column if it's the query itself)
        nn_perts = np.array([sig2pert[sig_id_order[i]] 
                            for i in indices[:, 1:k+1].flatten()]) \
                .reshape(len(sig_pert), k)
        
        # Check if ground truth is in the top-k
        correct = np.any(nn_perts == sig_pert[:, None], axis=1)
        
        precision = np.mean(correct) * 100
        precisions[f"precision@{k}"] = precision

    return precisions

In [ ]:
models = ['original', 'FT', 'selected']
source = ['GEx', 'PCA', 'scGPT']


all_precisions = {}
for s in source:
    for m in tqdm(models):
        print(f'Processing {s} - {m}')
        output_path = path_save + 'cp_nn_distances_' + s + '_' + m + '.h5'
        with h5py.File(output_path, 'r') as data_file:
            indices = data_file['indices'][:]
        
        sig_id_order = pickle.load(open(path_save + 'order_sig_' + s + '_' + m + '.pkl', 'rb'))
        
        precisions = overall_hit(sig_id_order, indices, k=[1, 5, 10, 100])
        all_precisions[f'{s}_{m}'] = precisions




In [ ]:
################### figure 2F ####################

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ----- INPUT: your results dict -----
results = all_precisions



# ----- Map keys to (source, model) -----
source_map = {'GEx': 'GEx', 'PCA': 'PCA', 'scGPT': 'scGPT'}
model_map  = {'original': 'Pre-trained', 'FT': 'Fine-tuned', 'selected': 'Fine-tuned HQ'}

rows = []
pat = re.compile(r'^(GEx|PCA|scGPT)_(original|FT|selected)$', re.I)
for k, metrics in results.items():
    m = pat.match(k)
    if not m:
        continue
    src_key, mdl_key = m.group(1), m.group(2)
    row = {'source': source_map[src_key], 'model': model_map[mdl_key]}
    row.update(metrics)
    rows.append(row)

df = pd.DataFrame(rows)

# ----- Ordering and palette -----
model_order  = ['Pre-trained', 'Fine-tuned', 'Fine-tuned HQ']
source_order = ['GEx', 'PCA', 'scGPT']
source_palette  = {
    "GEx": "#FF6961",                 # pastel red
    "PCA": "#77DD77",                 # pastel green
    "scGPT": sns.color_palette("pastel")[9]  # pastel violet
}

df_plot = df.copy()
df_plot['model']  = pd.Categorical(df_plot['model'],  categories=model_order,  ordered=True)
df_plot['source'] = pd.Categorical(df_plot['source'], categories=source_order, ordered=True)
df_plot = df_plot.sort_values(['model','source'])

metrics = ['precision@100']
titles  = {'precision@100':'Hit@100'}

fig, axes = plt.subplots(figsize=(4, 4))

group_gap = 1.2
offsets = np.linspace(-0.2, 0.2, len(source_order))

present_sources = [s for s in source_order if s in df_plot['source'].unique()]
legend_elems = [Line2D([0],[0], marker='o', linestyle='',
                       markerfacecolor=source_palette[s], markeredgecolor='black',
                       label=s) for s in present_sources]

xs, ys, colors = [], [], []

for mi, m in enumerate(model_order):
    base = mi * group_gap
    for si, s in enumerate(source_order):
        row = df_plot[(df_plot['model'] == m) & (df_plot['source'] == s)]
        if row.empty or metrics[0] not in row.columns:
            continue
        x = base + offsets[si]
        y = float(row[metrics[0]].iloc[0])  # already in %
        xs.append(x); ys.append(y); colors.append(source_palette[s])

# stems
for x, y in zip(xs, ys):
    axes.vlines(x, 0, y, color='black', linewidth=1)
# dots
axes.scatter(xs, ys, c=colors, s=70, edgecolor='black', zorder=3)

# labels and grid
axes.set_xticks([mi * group_gap for mi in range(len(model_order))])
axes.set_xticklabels(model_order)
axes.set_ylim(0, 105)
axes.grid(axis='y', linestyle='--', zorder=0, alpha=0.6)
axes.set_title('Compound recovery (Hit@100)')
axes.set_ylabel('Hit (%)')

# one legend
plt.legend(handles=legend_elems,loc = 'upper left', frameon=False)





# Final figure assembly

In [ ]:
# ================================
# Figure 2 — consistent with Fig 4/5
#   A: UMAPs (3 panels)
#   B: Distance KDEs (3 panels)
#   C: Intracoherence (3 panels)
#   D/E/F: right column (cell ARI / pert ARI / Hit)
# ================================
import re
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb, LinearSegmentedColormap
from matplotlib.lines import Line2D

# -------------------------
# Global style (match Fig 4/5)
# -------------------------
TITLE_FS  = 11
LABEL_FS  = 10
TICK_FS   = 9
LETTER_FS = 12
ROW_FS    = 12  # row-section title

def add_panel_letter(ax, letter, x=-0.12, y=1.08):
    ax.text(x, y, letter, transform=ax.transAxes,
            fontsize=LETTER_FS, va="top", ha="left")

def add_row_title(ax_left, text):
    """Put a row title above the left-most axis of that row."""
    ax_left.text(0.0, 1.18, text, transform=ax_left.transAxes,
                 fontsize=ROW_FS, va="bottom", ha="left")

# ================================
# Figure & Grid (3 rows × 4 cols)
# ================================
fig = plt.figure(figsize=(10.7, 8), dpi=300, facecolor="white", constrained_layout=True)
gs = fig.add_gridspec(
    3, 4,
    width_ratios=[1, 1, 1, 0.95]
)

# Axes
axA0 = fig.add_subplot(gs[0, 0]); axA1 = fig.add_subplot(gs[0, 1]); axA2 = fig.add_subplot(gs[0, 2]); axD = fig.add_subplot(gs[0, 3])
axB0 = fig.add_subplot(gs[1, 0]); axB1 = fig.add_subplot(gs[1, 1]); axB2 = fig.add_subplot(gs[1, 2]); axE = fig.add_subplot(gs[1, 3])
axC0 = fig.add_subplot(gs[2, 0]); axC1 = fig.add_subplot(gs[2, 1]); axC2 = fig.add_subplot(gs[2, 2]); axF = fig.add_subplot(gs[2, 3])

# ================================
# SECTION LETTERS (A–F)
# ================================
# A/B/C letters on the first subplot of each left-block row
add_panel_letter(axA0, "A")
add_panel_letter(axB0, "B")
add_panel_letter(axC0, "C")

# D/E/F letters on the right column panels
add_panel_letter(axD, "D")
add_panel_letter(axE, "E")
add_panel_letter(axF, "F")

# ================================
# Row titles (A/B/C) — like Fig 4/5
# ================================
fig.text(0.5, 1.2, 'UMAP of scGPT Embeddings', ha='center', va='top', fontsize= TITLE_FS)
fig.text(0.5, 0.65, 'Pairwise Cosine Distance Distributions', ha='center', va='top', fontsize=TITLE_FS)
fig.text(0.5, 0.33, 'Intra-perturbation Label Agreement', ha='center', va='top', fontsize=TITLE_FS)

# ================================
# Row A: UMAPs
# ================================
umap_panels = [
    (axA0, adata_original, f"Pre-trained scGPT\n{adata_original.shape[0]} embeddings"),
    (axA1, adata_FT,       f"Fine-tuned scGPT\n{adata_FT.shape[0]} embeddings"),
    (axA2, adata_selected, f"Fine-tuned scGPT (HQ)\n{adata_selected.shape[0]} embeddings"),
]
for ax, adata, title in umap_panels:
    ax.scatter(
        adata.obsm["X_umap"][:, 0], adata.obsm["X_umap"][:, 1],
        color=adata.obs.color, s=0.01, alpha=0.2
    )
    ax.set_title(title, fontsize=TITLE_FS)
    ax.axis("off")

# ================================
# Row B: KDE distance distributions
# ================================
C_SAMECELL = sns.color_palette("pastel")[4]
C_SAMEPERT = sns.color_palette("pastel")[0]
C_RANDOM   = sns.color_palette("pastel")[1]

distance_sets = [
    (axB0, distances_original["Same cell"], distances_original["Same pert"], distances_original["Diff cell / pert"]),
    (axB1, distances_FT["Same cell"],       distances_FT["Same pert"],       distances_FT["Diff cell / pert"]),
    (axB2, distances_selected["Same cell"], distances_selected["Same pert"], distances_selected["Diff cell / pert"]),
]

for i, (ax, dist_cell, dist_drug, dist_rand) in enumerate(distance_sets):
    sns.kdeplot(dist_cell, label="Same cell", color=C_SAMECELL, ax=ax, fill=True, alpha=0.8, linewidth=2, zorder=2)
    sns.kdeplot(dist_drug, label="Same perturbation", color=C_SAMEPERT, ax=ax, fill=True, alpha=0.8, linewidth=2, zorder=2)
    sns.kdeplot(dist_rand, label="Different cell / perturbation", color=C_RANDOM, ax=ax, fill=True, alpha=0.8, linewidth=2, zorder=2)

    ax.set_xlabel("Cosine distance", fontsize=LABEL_FS)
    ax.set_title(["Pre-trained", "Fine-tuned", "Fine-tuned HQ"][i], fontsize=TITLE_FS)
    if i == 0:
        ax.set_ylabel("Density", fontsize=LABEL_FS)
    else:
        ax.set_ylabel("")
    ax.tick_params(labelsize=TICK_FS)

    ax.grid(alpha=0.3, ls="--", zorder=0)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)

# keep only one KDE legend (center panel)
axB0.legend([], [], frameon=False)
axB2.legend([], [], frameon=False)
axB1.legend([], [], frameon=False)

# ================================
# Row C: Intracoherence (precomputed)
# ================================
rowC_axes = [
    (axC0, "original", "GEx vs Pre-trained"),
    (axC1, "FT",       "GEx vs Fine-tuned"),
    (axC2, "selected", "GEx HQ vs Fine-tuned HQ"),
]

norm = plt.Normalize(vmin=0, vmax=5)
base = np.array(to_rgb(sns.color_palette("pastel")[0]))
stronger = np.clip(base * 0.8, 0, 1)
stronger = stronger / np.max(stronger) ** 0.7
custom_cmap = LinearSegmentedColormap.from_list(
    "white_to_stronger_pastel0", ["#FFFFFF", stronger]
)

last_sc = None
for ax, key, title in rowC_axes:
    d = intra_data[key]

    sc = ax.scatter(
        d["x"], d["y"],
        s=np.clip(d["sizes"], 10, 200) * 1.5,
        c=d["colors"], cmap=custom_cmap, norm=norm,
        alpha=0.9,
        # linewidth=d["edge_widths"], 
        # edgecolor="black"
    )

    ax.set_axisbelow(True)
    ax.axvline(0, color="gray", linestyle="--", lw=1, zorder=1)
    ax.grid(alpha=0.3, ls="--", zorder=0)

    ax.set_title(title, fontsize=TITLE_FS)
    ax.set_xlabel("Δ Avg Distance (GEx – scGPT)", fontsize=LABEL_FS)
    if ax is axC0:
        ax.set_ylabel("Max Distance", fontsize=LABEL_FS)
    else:
        ax.set_ylabel("")
    ax.tick_params(labelsize=TICK_FS)

    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)

    last_sc = sc

# ================================
# Right column: D/E ARI bars + F Hit@100
# ================================
hue_order = ["GEx", "PCA", "scGPT"]
source_palette = {
    "GEx":  sns.color_palette("pastel")[3],
    "PCA":  sns.color_palette("pastel")[2],
    "scGPT": sns.color_palette("pastel")[0],
}
model_rename = {"Original": "Pre-trained", "FT": "Fine-tuned", "Selected": "Fine-tuned HQ"}

def style_bar_ax(ax, title):
    ax.set_title(title, fontsize=TITLE_FS)
    ax.set_ylabel("Adjusted Rand Index", fontsize=LABEL_FS)
    ax.set_xlabel("Model", fontsize=LABEL_FS)
    ax.tick_params(labelsize=TICK_FS)
    ax.grid(axis="y", linestyle="--", alpha=0.6, zorder=0)
    ax.set_axisbelow(True)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)

# --- D: cell_id ARI ---
subset = ari[ari["label"] == "cell_iname"].copy()
subset["source"] = subset["source"].astype(str).replace({"emb": "scGPT"})
subset["model"]  = subset["model"].astype(str).replace(model_rename)

sns.barplot(
    data=subset, x="model", y="ARI", hue="source",
    hue_order=hue_order, palette=source_palette,
    edgecolor="black", ax=axD
)
axD.set_ylim(0, subset["ARI"].max() * 1.2)
style_bar_ax(axD, "Clustering performance by cell")
axD.legend(title="Data type", fontsize=8, title_fontsize=9, frameon=False, loc="best")

# --- E: pert_id ARI ---
subset = ari[ari["label"] == "pert_id"].copy()
subset["source"] = subset["source"].astype(str).replace({"emb": "scGPT"})
subset["model"]  = subset["model"].astype(str).replace(model_rename)

sns.barplot(
    data=subset, x="model", y="ARI", hue="source",
    hue_order=hue_order, palette=source_palette,
    edgecolor="black", ax=axE
)
axE.set_ylim(0, subset["ARI"].max() * 1.2)
style_bar_ax(axE, "Clustering performance by perturbation")
axE.legend([], [], frameon=False)  # hide duplicate legend

# --- F: Hit@100 (stem+dot) ---

results = all_precisions

source_map = {"GEx": "GEx", "PCA": "PCA", "scGPT": "scGPT"}
model_map  = {"original": "Pre-trained", "FT": "Fine-tuned", "selected": "Fine-tuned HQ"}

rows = []
pat = re.compile(r"^(GEx|PCA|scGPT)_(original|FT|selected)$", re.I)
for k, metrics in results.items():
    m = pat.match(k)
    if not m:
        continue
    src_key, mdl_key = m.group(1), m.group(2)
    row = {"source": source_map[src_key], "model": model_map[mdl_key]}
    row.update(metrics)
    rows.append(row)

df = pd.DataFrame(rows)

model_order  = ["Pre-trained", "Fine-tuned", "Fine-tuned HQ"]
source_order = ["GEx", "PCA", "scGPT"]

df_plot = df.copy()
df_plot["model"]  = pd.Categorical(df_plot["model"],  categories=model_order,  ordered=True)
df_plot["source"] = pd.Categorical(df_plot["source"], categories=source_order, ordered=True)
df_plot = df_plot.sort_values(["model", "source"])

metric = "precision@100"
group_gap = 1.2
offsets = np.linspace(-0.2, 0.2, len(source_order))

legend_elems = [
    Line2D([0],[0], marker="o", linestyle="",
           markerfacecolor=source_palette[s], markeredgecolor="black",
           label=s)
    for s in source_order
]

xs, ys, cols = [], [], []
for mi, mname in enumerate(model_order):
    base_x = mi * group_gap
    for si, sname in enumerate(source_order):
        row = df_plot[(df_plot["model"] == mname) & (df_plot["source"] == sname)]
        if row.empty or metric not in row.columns:
            continue
        xs.append(base_x + offsets[si])
        ys.append(float(row[metric].iloc[0]))
        cols.append(source_palette[sname])

for x, y in zip(xs, ys):
    axF.vlines(x, 0, y, color="black", linewidth=1, zorder=1)
axF.scatter(xs, ys, c=cols, s=70, edgecolor="black", zorder=3)

axF.set_xticks([mi * group_gap for mi in range(len(model_order))])
axF.set_xticklabels(model_order)
axF.set_ylim(0, 105)
axF.set_title("Compound recovery (Hit@100)", fontsize=TITLE_FS)
axF.set_ylabel("Hit (%)", fontsize=LABEL_FS)
axF.tick_params(labelsize=TICK_FS)
axF.grid(axis="y", linestyle="--", alpha=0.6, zorder=0)
axF.set_axisbelow(True)
axF.spines["right"].set_visible(False)
axF.spines["top"].set_visible(False)
axF.legend(handles=legend_elems, loc="upper left", frameon=False, fontsize=8, title="Data type", title_fontsize=9)

# ================================
# Colorbar for intracoherence (Row C)
# ================================
if last_sc is not None:
    cbar_ax = fig.add_axes([0.25, 0.0, 0.5, 0.02])  # [left, bottom, width, height]
    cbar = plt.colorbar(last_sc, cax=cbar_ax, orientation="horizontal")
    cbar.set_label("-log10(p-value)", fontsize=LABEL_FS)
    cbar.ax.tick_params(labelsize=TICK_FS)

handles, labels = axB0.get_legend_handles_labels()
legend = fig.legend(handles, labels, loc='upper center', ncol=3, frameon=False, fontsize=10, bbox_to_anchor=(0.5, 0.625))
for h in legend.legend_handles:
    try:
        h.set_linewidth(1)
    except Exception:
        pass

plt.show()


